# LUTorch MNIST with `ProjectionLUT`

This notebook trains a simple MNIST classifier using the LUTorch `ProjectionLUT`
module (built on top of `MultiHeadLut`) instead of the legacy fused `ProjectionLUTLayer`.


In [ ]:
import os
import time
import logging
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.transforms.functional import to_pil_image
from tqdm.notebook import tqdm

from spiky.lutorch.multi_head_lut import ProjectionLUT, UnfoldConfiguration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
random_seed = 1
torch.manual_seed(random_seed)
np.random.seed(random_seed)
torch.backends.cudnn.enabled = True

logging.basicConfig(level=logging.INFO)
logging.info(f'Using device: {device}')


## MNIST data preparation

We load MNIST and keep it as dense tensors on the selected device.


In [ ]:
mnist_dataset_dir = 'mnist'

mnist_train_dataset_raw = torchvision.datasets.MNIST(
    mnist_dataset_dir, train=True, download=True
)
mnist_test_dataset_raw = torchvision.datasets.MNIST(
    mnist_dataset_dir, train=False, download=True
)

mnist_train_data = mnist_train_dataset_raw.data.to(device=device, dtype=torch.float32) / 255.0
mnist_test_data = mnist_test_dataset_raw.data.to(device=device, dtype=torch.float32) / 255.0
mnist_train_targets = mnist_train_dataset_raw.targets.to(device=device)
mnist_test_targets = mnist_test_dataset_raw.targets.to(device=device)

mnist_train_dataset = TensorDataset(mnist_train_data, mnist_train_targets)
mnist_test_dataset = TensorDataset(mnist_test_data, mnist_test_targets)

batch_size = 128
train_loader = DataLoader(mnist_train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test_dataset, batch_size=batch_size, shuffle=False)

# Quick visual sanity check
example_batch, example_targets = next(iter(train_loader))
to_pil_image(example_batch[0].cpu())


## LUTorch model using `ProjectionLUT`

We define a simple model:
- One `ProjectionLUT` operating on the 2D MNIST image (28×28).
- A linear classifier on top of the flattened LUT outputs.

The unfold configuration uses non-overlapping 4×4 patches (stride 4), which yields
a 7×7 patch grid. Each patch produces `O` output channels.


In [ ]:
input_H, input_W = 28, 28
num_classes = 10

unfold_cfg = UnfoldConfiguration(
    H=input_H,
    W=input_W,
    kernel_size=4,
    stride=4,
)

class LUTorchMNIST(nn.Module):
    def __init__(self, device):
        super().__init__()

        # Each patch will produce O features.
        O = 8
        n_anchor_pairs = 4
        tables_per_head = 1

        self.proj = ProjectionLUT(
            unfold_config=unfold_cfg,
            O=O,
            n_anchor_pairs=n_anchor_pairs,
            tables_per_head=tables_per_head,
            device=torch.device(device),
        )

        H_p, W_p = self.proj.H_p, self.proj.W_p
        self.H_p, self.W_p, self.O = H_p, W_p, O

        self.classifier = nn.Linear(H_p * W_p * O, num_classes, device=device)

    def forward(self, x):
        # x: [B, 28, 28]
        z = self.proj(x)  # [B, H_p, W_p, O]
        z = z.view(x.shape[0], -1)
        return self.classifier(z)

model = LUTorchMNIST(device).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## Training loop


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # images: [B, 28, 28]
        logits = model(images)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = logits.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}: loss={epoch_loss:.4f}, acc={epoch_acc:.4f}")


## Evaluation on test set


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, targets in test_loader:
        logits = model(images)
        _, predicted = logits.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

test_acc = correct / total
print(f"Test accuracy: {test_acc:.4f}")
